# Week 1 · ZoroLogistics Data Generator：毎週再利用するdataset

# Requirements: pip install numpy pandas
# Uses: repo rootの`zoro` package（`from zoro import data`）。

このnotebookは`zoro/data.py`のseed付きgeneratorを実行し、**100,000行**のdatasetを`data/`に保存し、各columnの意味を見知らぬ人に伝える **data dictionary** を生成します。dataには意図的に *flaw*（duplicate、`NaN` weight、`NaN` distance）を入れているため、Week 2で本物のcleaning作業ができます。

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root
p = pathlib.Path.cwd()
while not (p / "zoro").is_dir() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

from zoro import data
print("zoro.data loaded:", data.__name__)

### case studyを1段落で

**ZoroLogistics**は架空のfreight companyです。Zorostが実際に関わる、規制とtraceabilityを重視するindustryをモデルにしています。その世界は、3つのmaster tableである **carriers**（freightを運ぶ会社）、**lanes**（都市間のroute）、**shipments**（何がいつ動き、on timeで届いたか）に加え、LLM週で使うsupport tickets、policy documents、bills of ladingから構成されます。すべてdeterministicに生成されるため、seedは *contract* です：同じseed、同じcompanyになります。

In [ ]:
# A small deterministic preview of the three core tables.
carriers_df = data.carriers(20, seed=7)
lanes_df    = data.lanes(20, seed=11)
ship_df     = data.shipments(2_000, seed=42)

for name, df in [("carriers", carriers_df), ("lanes", lanes_df), ("shipments", ship_df)]:
    print(f"{name:<10} {df.shape[0]:>6,} rows x {df.shape[1]} cols")

### schemaを読む

100k行を生成する前に、まず少量を見てみましょう。`dtypes`はphysical typeを、`head()`はbusiness上のshapeを示します。`shipments`がdenormalized textではなくforeign key（`carrier_id`、`lane_id`）を持つことに注目してください。このjoin構造をWeek 2のSQLで利用します。

In [ ]:
import pandas as pd
print("carriers dtypes:")
print(carriers_df.dtypes)
print()
print(carriers_df.head(3).to_string())
print()
print("shipments numeric summary:")
print(ship_df[["weight_kg", "value_usd", "delay_hours"]].describe().round(2).to_string())

### 全datasetを生成して保存する

`save_all()`は固定seedのもとですべてのgeneratorを実行し、CSV fileを書き出します。output directoryは **notebook folderではなくrepo root** を基準にresolveされます。そのためdataはprogramが、そしてあなたのforkのhistoryが期待する`data/`に置かれます。

In [ ]:
repo = pathlib.Path.cwd()
while not (repo / "zoro").is_dir() and repo != repo.parent:
    repo = repo.parent

saved = data.save_all(out_dir=str(repo / "data"), seed=42, n=100_000)
for name, df in saved.items():
    print(f"{name:<10} {len(df):>9,} rows")
print("Saved CSV files to", str(repo / "data"))

### data dictionary

data dictionaryはschemaを **人間に読める形** にしたものです。table、dtype、freight上の意味を1columnにつき1rowで記載します。これは、見知らぬ人がdataを読む前に確認する成果物です。`data/data-dictionary.md`に書き出します。

In [ ]:
dictionary = [
    ("carriers", "carrier_id", "object", "Primary key; e.g. C001..C020"),
    ("carriers", "carrier_name", "object", "Display name (Atlas Freight, BlueHarbor, ...)"),
    ("carriers", "region", "object", "Operating region: North/South/East/West/Central"),
    ("carriers", "on_time_rate", "float64", "Reliability 0.60-0.99; drives lateness probability"),
    ("carriers", "base_rate_usd_per_km_ton", "float64", "USD per km per ton, 0.6-2.6"),
    ("carriers", "fleet_size", "int64", "Number of vehicles, 50-4000"),
    ("lanes", "lane_id", "object", "Primary key; e.g. L001..L020"),
    ("lanes", "origin", "object", "Origin city"),
    ("lanes", "destination", "object", "Destination city"),
    ("lanes", "distance_km", "float64", "Route distance; has planted NaNs (Week 2 finds them)"),
    ("lanes", "avg_transit_days", "float64", "Expected transit in days"),
    ("lanes", "toll_km", "int64", "Kilometers of toll road on the lane"),
    ("lanes", "port_region", "object", "Nearest port"),
    ("shipments", "shipment_id", "object", "Primary key; S0000001.."),
    ("shipments", "carrier_id", "object", "FK -> carriers.carrier_id"),
    ("shipments", "lane_id", "object", "FK -> lanes.lane_id"),
    ("shipments", "commodity", "object", "Cargo type (electronics, apparel, ...)"),
    ("shipments", "weight_kg", "float64", "Weight; has planted NaNs"),
    ("shipments", "value_usd", "float64", "Declared value"),
    ("shipments", "planned_departure", "datetime64[ns]", "Scheduled pickup"),
    ("shipments", "planned_arrival", "datetime64[ns]", "Promised delivery"),
    ("shipments", "actual_arrival", "datetime64[ns]", "Real delivery"),
    ("shipments", "delay_hours", "float64", "actual minus planned in hours; the Week 3-4 target"),
    ("shipments", "is_on_time", "bool", "delay_hours <= 2.0 grace window"),
    ("shipments", "status", "object", "Delivered / In Transit / Booked"),
    ("shipments", "weather_severity", "object", "clear/light/moderate/severe at dispatch"),
]
dd = pd.DataFrame(dictionary, columns=["table", "column", "dtype", "meaning"])
print(dd.to_string(index=False))

lines = ["# ZoroLogistics Data Dictionary", "", "| table | column | dtype | meaning |", "|---|---|---|---|"]
for t, col, dt, m in dictionary:
    lines.append(f"| {t} | {col} | {dt} | {m} |")
(repo / "data" / "data-dictionary.md").write_text("\n".join(lines) + "\n")
print()
print("Wrote data/data-dictionary.md with", len(dictionary), "column definitions.")

### 保存を確認し、Week 2の仕事をpreviewする

CSVをreloadし、row countをassertし、Week 2が見つけて直す **planted data-quality issue** を表示します。duplicate shipment row、`NaN` weight、`NaN` lane distanceです。先に見ておけば、cleaning週は驚きではなくhuntになります。

In [ ]:
import numpy as np
raw_carriers = pd.read_csv(repo / "data" / "carriers.csv")
raw_lanes    = pd.read_csv(repo / "data" / "lanes.csv")
raw_ship     = pd.read_csv(repo / "data" / "shipments.csv", parse_dates=["planned_departure", "planned_arrival", "actual_arrival"])
raw_tickets  = pd.read_csv(repo / "data" / "support_tickets.csv")

assert len(raw_carriers) == 20
assert len(raw_lanes) == 20
# zoro.data plants ~0.2% duplicate rows on purpose (Week 2 finds them),
# so the count is slightly above the requested n.
assert 100_000 <= len(raw_ship) <= 101_000, f"unexpected row count: {len(raw_ship)}"
assert len(raw_tickets) == 2_000

n_dupes      = int(raw_ship.duplicated().sum())
n_nan_weight = int(raw_ship["weight_kg"].isna().sum())
n_nan_dist   = int(raw_lanes["distance_km"].isna().sum())
print(f"duplicate shipment rows   : {n_dupes:,}")
print(f"NaN weight_kg (shipments) : {n_nan_weight:,}")
print(f"NaN distance_km (lanes)   : {n_nan_dist:,}")
print("(Week 2 will find and fix all three.)")

### metric

最後は1つの数字で終えます。すべてのtableで生成・保存されたrowの合計です。

In [ ]:
total_rows = len(raw_carriers) + len(raw_lanes) + len(raw_ship) + len(raw_tickets)
print("TOTAL_ROWS_GENERATED:", total_rows)